# Phase 4 - Step 5: DR Grading (5 0 grading dataset preparationV2)


In [ ]:
!pip install ultralytics segmentation-models-pytorch albumentations -q


## Step 2: Hiding the Data to Stop Kaggle Freezing!
Kaggle's interface freezes when it tries to render 50,000+ files in the `/working/` sidebar. 
We will bypass this by saving everything to the invisible `/kaggle/temp/` folder, and then ZIP it back securely at the very end!


In [ ]:
import os, cv2, glob, torch, shutil
import numpy as np
from tqdm import tqdm
from PIL import Image
import segmentation_models_pytorch as smp
from ultralytics import YOLO
import torch.nn.functional as F
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_IMG_DIR = '/kaggle/input/datasets/mhshanto007/oversampled-dataset/Oversampling Dataset 3.0'

# Save to TEMP folder so the Kaggle Website doesn't crash!
OUT_DIR = '/kaggle/temp/phase_b_dataset'
os.makedirs(OUT_DIR, exist_ok=True)

# Clean, separated folder structure
CHANNELS = ['images', 'vessel', 'MA', 'HE', 'EX', 'OD', 'CW']
for ch in CHANNELS:
    for grade in ['0_No_DR', '1_Mild', '2_Moderate', '3_Severe', '4_Proliferative_DR']:
        os.makedirs(os.path.join(OUT_DIR, ch, grade), exist_ok=True)



In [ ]:
import torch
import torch.nn as nn
class ChannelAttention(nn.Module):
    """
    Channel Attention Module (Squeeze-and-Excitation style).
    Learns per-channel importance weights via global average pooling.
    """
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


class SpatialAttention(nn.Module):
    """
    Spatial Attention Module.
    Learns a 2D attention map highlighting important spatial locations.
    Uses max-pooled and avg-pooled features across the channel dimension.
    """
    def __init__(self, kernel_size=7):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        combined = torch.cat([avg_out, max_out], dim=1)
        attn = self.conv(combined)
        return x * attn


class CSBlock(nn.Module):
    """
    Channel-Spatial Attention Block.
    Applies Channel Attention → Spatial Attention sequentially.
    """
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention()

    def forward(self, x):
        x = self.ca(x)   # Channel: "Which features matter?"
        x = self.sa(x)   # Spatial: "Where to look?"
        return x


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)


class CSNet(nn.Module):
    """CS-Net: U-Net with Channel-Spatial Attention at every encoder and skip level."""
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super().__init__()

        # Encoder path
        self.encoders = nn.ModuleList()
        self.cs_blocks = nn.ModuleList()
        self.pools = nn.ModuleList()

        ch = in_channels
        for f in features:
            self.encoders.append(ConvBlock(ch, f))
            self.cs_blocks.append(CSBlock(f))
            self.pools.append(nn.MaxPool2d(2))
            ch = f

        # Bottleneck
        self.bottleneck = ConvBlock(features[-1], features[-1] * 2)
        self.cs_bottleneck = CSBlock(features[-1] * 2)

        # Decoder path
        self.upconvs = nn.ModuleList()
        self.decoders = nn.ModuleList()

        rev = list(reversed(features))
        prev = features[-1] * 2
        for f in rev:
            self.upconvs.append(nn.ConvTranspose2d(prev, f, 2, stride=2))
            self.decoders.append(ConvBlock(f * 2, f))
            prev = f

        self.final = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x):
        skips = []
        for enc, cs, pool in zip(self.encoders, self.cs_blocks, self.pools):
            x = enc(x)
            x = cs(x)     # Apply CS attention
            skips.append(x)
            x = pool(x)

        x = self.bottleneck(x)
        x = self.cs_bottleneck(x)

        skips = skips[::-1]
        for up, dec, skip in zip(self.upconvs, self.decoders, skips):
            x = up(x)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)

        return self.final(x)
class DecoderBlock(nn.Module):
    """Decoder block: upsample + concat skip + convolutions."""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.up(x)
        # Handle size mismatch
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class SwinUNet(nn.Module):
    def __init__(self, out_channels=1, pretrained=True):
        super().__init__()

        self.encoder = timm.create_model(
            'swin_tiny_patch4_window7_224',
            pretrained=pretrained,
            features_only=True,
            img_size=512,
        )

        # Store expected channels: [96, 192, 384, 768]
        self.encoder_channels = self.encoder.feature_info.channels()
        print(f'Encoder channels per stage: {self.encoder_channels}')

        # Decoder
        self.dec4 = DecoderBlock(self.encoder_channels[3], self.encoder_channels[2], 256)
        self.dec3 = DecoderBlock(256, self.encoder_channels[1], 128)
        self.dec2 = DecoderBlock(128, self.encoder_channels[0], 64)

        self.final_up = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 4, stride=4),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.final_conv = nn.Conv2d(32, out_channels, 1)

    def _fix_feature_format(self, features):
        """
        timm's Swin outputs (B, H, W, C) — channels-last.
        PyTorch Conv2d needs (B, C, H, W) — channels-first.

        We check each feature: if dim[1] != expected channels, permute it.
        """
        fixed = []
        for feat, expected_c in zip(features, self.encoder_channels):
            if feat.shape[1] != expected_c:
                feat = feat.permute(0, 3, 1, 2).contiguous()
            fixed.append(feat)
        return fixed

    def forward(self, x):
        features = self.encoder(x)

        # Fix format: (B, H, W, C) → (B, C, H, W)
        features = self._fix_feature_format(features)

        # Decoder with skip connections
        d4 = self.dec4(features[3], features[2])
        d3 = self.dec3(d4, features[1])
        d2 = self.dec2(d3, features[0])

        out = self.final_up(d2)
        out = F.interpolate(out, size=(x.shape[2], x.shape[3]), mode='bilinear', align_corners=False)
        return self.final_conv(out)




## Step 3: Load Stage 1 Models


In [ ]:
CSNET_VESSEL = '/kaggle/input/datasets/aymenslimani/retinal-stage-1-models/csnet_best.pt'
ATTN_UNET_VESSEL = '/kaggle/input/datasets/aymenslimani/retinal-stage-1-models/attentionUNet_best_model.pt'
SWIN_UNET_VESSEL = '/kaggle/input/datasets/aymenslimani/retinal-stage-1-models/swin_unet_best.pt'
ATTN_UNET_LESION = '/kaggle/input/datasets/aymenslimani/retinal-stage-1-models/Attention_UNet_ResNet34_best.pth'
UNETPP_LESION = '/kaggle/input/datasets/aymenslimani/retinal-stage-1-models/UNetPP_ResNet34_best.pth'
YOLO_LESION = '/kaggle/input/datasets/aymenslimani/retinal-stage-1-models/YOLOV11_best.pt'

vessel_csnet = CSNet(out_channels=1).to(device)
if os.path.exists(CSNET_VESSEL): vessel_csnet.load_state_dict(torch.load(CSNET_VESSEL, map_location=device, weights_only=False)['model_state_dict'])
vessel_csnet.eval()

vessel_attn = smp.Unet(encoder_name='resnet34', decoder_attention_type='scse', in_channels=3, classes=1).to(device)
if os.path.exists(ATTN_UNET_VESSEL): vessel_attn.load_state_dict(torch.load(ATTN_UNET_VESSEL, map_location=device, weights_only=False)['model_state_dict'])
vessel_attn.eval()

vessel_swin = SwinUNet(out_channels=1).to(device)
if os.path.exists(SWIN_UNET_VESSEL): vessel_swin.load_state_dict(torch.load(SWIN_UNET_VESSEL, map_location=device, weights_only=False)['model_state_dict'])
vessel_swin.eval()

attn_unet_lesions = smp.Unet(encoder_name='resnet34', encoder_weights=None, in_channels=3, classes=5).to(device)
if os.path.exists(ATTN_UNET_LESION): attn_unet_lesions.load_state_dict(torch.load(ATTN_UNET_LESION, map_location=device, weights_only=False)['model_state_dict'])
attn_unet_lesions.eval()

unet_pp = smp.Unet(encoder_name='resnet34', encoder_weights=None, in_channels=3, classes=5).to(device)
if os.path.exists(UNETPP_LESION): unet_pp.load_state_dict(torch.load(UNETPP_LESION, map_location=device, weights_only=False)['model_state_dict'])
unet_pp.eval()

yolo_model = YOLO(YOLO_LESION) if os.path.exists(YOLO_LESION) else None



## Step 4: The Generator (Streamlined Folders)


In [ ]:

def crop_roi(img, tol=7):
    if img.ndim == 2: mask = img > tol
    elif img.ndim == 3: mask = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) > tol
    m, n = mask.shape
    mask0, mask1 = mask.any(0), mask.any(1)
    if not mask0.any() or not mask1.any(): return img
    col_start, col_end = mask0.argmax(), n - mask0[::-1].argmax()
    row_start, row_end = mask1.argmax(), m - mask1[::-1].argmax()
    return img[row_start:row_end, col_start:col_end]

def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l_channel, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l_channel)
    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)

def preprocess_image(img_path, size=512):
    img = cv2.imread(img_path)
    if img is None: return None, None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # EXACT STAGE 1 PREPROCESSING
    img = crop_roi(img)
    img = apply_clahe(img)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_LINEAR)
    
    tensor = img.astype(np.float32) / 255.0
    mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
    tensor = (tensor - mean) / std
    return img, torch.tensor(tensor).permute(2, 0, 1).unsqueeze(0).float().to(device)




def get_yolo_masks(img_array, size=512):
    prob_yolo = np.zeros((5, size, size), dtype=np.float32)
    if yolo_model is None: return prob_yolo
    # Pass the exactly squared 512x512 array so YOLO maintains strict dimensions
    res = yolo_model.predict(img_array, imgsz=size, verbose=False, retina_masks=True)[0]
    if res.masks is not None:
        masks_data = res.masks.data.cpu().numpy()
        classes = res.boxes.cls.cpu().numpy().astype(int)
        for idx, cls_id in enumerate(classes):
            if cls_id < 5: 
                mask = masks_data[idx]
                # Fallback reshape just in case YOLO letterboxes it
                if mask.shape != (size, size): mask = cv2.resize(mask, (size, size))
                prob_yolo[cls_id] = np.maximum(prob_yolo[cls_id], mask)
    return prob_yolo

image_paths = []
for ext in ('*.png', '*.jpg', '*.jpeg', '*.tif', '*.tiff'):
    image_paths.extend(glob.glob(os.path.join(INPUT_IMG_DIR, '**', ext), recursive=True))

with torch.no_grad():
    for img_path in tqdm(image_paths, desc="Generating Phase B Masks"):
        rel_path = os.path.relpath(img_path, INPUT_IMG_DIR)
        grade_folder = os.path.dirname(rel_path) # e.g. '0_No_DR'
        filename = os.path.basename(img_path)    # e.g. '10_left.png'
        
        orig_img, tensor_img = preprocess_image(img_path)
        if tensor_img is None: continue
        
        prob_v_csnet = torch.sigmoid(vessel_csnet(tensor_img)).cpu().numpy()[0, 0]
        prob_v_attn  = torch.sigmoid(vessel_attn(tensor_img)).cpu().numpy()[0, 0]
        prob_v_swin  = torch.sigmoid(vessel_swin(tensor_img)).cpu().numpy()[0, 0]
        vessel_prob = (0.4 * prob_v_csnet) + (0.4 * prob_v_attn) + (0.2 * prob_v_swin)
        
        prob_attn = torch.sigmoid(attn_unet_lesions(tensor_img)).cpu().numpy()[0]
        prob_unpp = torch.sigmoid(unet_pp(tensor_img)).cpu().numpy()[0]
        prob_yolo = get_yolo_masks(orig_img)
        lesion_prob = (0.4 * prob_attn) + (0.4 * prob_unpp) + (0.2 * prob_yolo)
        
        # Save into the clean, separated folders using the exact same filename
        Image.fromarray(orig_img).save(os.path.join(OUT_DIR, 'images', grade_folder, filename))
        Image.fromarray((vessel_prob * 255).astype(np.uint8)).save(os.path.join(OUT_DIR, 'vessel', grade_folder, filename))
        
        lesion_names = ['MA', 'HE', 'EX', 'OD', 'CW']
        for i, lt in enumerate(lesion_names):
            Image.fromarray((lesion_prob[i] * 255).astype(np.uint8)).save(os.path.join(OUT_DIR, lt, grade_folder, filename))



## Step 5: ZIP Compression
Once the generation is done, this zips the hidden temp folder into a single fast file in your `/kaggle/working/` directory so you can easily download it or feed it to the training notebook.


In [ ]:

print("Zipping the dataset into a single file... Note: Do not close browser yet!")
shutil.make_archive("/kaggle/working/Phase_B_9Channel_DatasetV2", 'zip', OUT_DIR)
print("✅ Done! File 'Phase_B_9Channel_DatasetV2.zip' is ready in /kaggle/working/!")



## Step 6: Visualize the 9-Channel Stack!
Let's prove the dataloader logic works and looks beautiful!

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# Point exactly to your generated directory
OUT_DIR = '/kaggle/temp/phase_b_dataset'
grades = ['0_No_DR', '1_Mild', '2_Moderate', '3_Severe', '4_Proliferative_DR']

folders = ['images', 'vessel', 'MA', 'HE', 'EX', 'OD', 'CW']
titles = ['RGB', 'Vessel', 'MA', 'HE', 'EX', 'OD', 'CW']

for grade in grades:
    img_folder_path = os.path.join(OUT_DIR, 'images', grade)
    
    # Safety check in case a grade folder happens to be empty or missing
    if not os.path.exists(img_folder_path): 
        continue
        
    # Grab all images in this grade and strictly select the first 2
    all_imgs = os.listdir(img_folder_path)
    sample_imgs = all_imgs[:2] 
    
    for sample_img_name in sample_imgs:
        rgb = cv2.imread(os.path.join(OUT_DIR, 'images', grade, sample_img_name))
        rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)

        fig, axes = plt.subplots(1, 7, figsize=(24, 4))

        axes[0].imshow(rgb)
        axes[0].set_title(f"1. Original RGB")
        axes[0].axis('off')

        for i in range(1, 7):
            mask = cv2.imread(os.path.join(OUT_DIR, folders[i], grade, sample_img_name), cv2.IMREAD_GRAYSCALE)
            axes[i].imshow(mask, cmap='magma')
            axes[i].set_title(f"{i+1}. {titles[i]} Channel")
            axes[i].axis('off')

        # Add a big bold title to clearly label the severity Grade
        plt.suptitle(f"Grade: {grade}  |  Image: {sample_img_name}", fontsize=18, fontweight='bold', color='darkred')
        plt.tight_layout()
        plt.subplots_adjust(top=0.85) # Prevents the title from overlapping with the images
        plt.show()


In [ ]:
from IPython.display import FileLink

# This generates a secure download link for your zip file!
FileLink(r'Phase_B_9Channel_DatasetV2.zip')
